In [19]:
!pip install anthropic

In [20]:
import os
from google.colab import userdata
from anthropic import Anthropic

# Access the API key from Colab secrets and set it as an environment variable
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
os.environ['ANTHROPIC_API_KEY'] = ANTHROPIC_API_KEY

client = Anthropic()
model = "claude-haiku-4-5"

In [31]:
import json

def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
        }
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def text_from_message(response):
    # Helper function to extract text from the response object
    # if it's a message from the Anthropic API
    if isinstance(response, Message):
        for block in response.content:
            if block.type == "text":
                return block.text
    return str(response) # Fallback for other types or if no text block is found


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    if tools:
        params["tools"] = tools

    message = client.messages.create(**params)
    return message

In [32]:
from datetime import datetime
from anthropic.types import ToolParam
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):

    if not date_format:
        raise ValueError("Date format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = ToolParam(
    {
  "name": "get_current_datetime",
  "description": "Returns the current date and/or time formatted according to the specified format string. Use this when you need to know the current date, time, or both. Defaults to 'YYYY-MM-DD HH:MM:SS' format if no format is provided.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A Python strftime-compatible format string that controls the output format. Common examples: '%Y-%m-%d' for date only (e.g. 2025-04-25), '%H:%M:%S' for time only (e.g. 14:30:00), '%Y-%m-%d %H:%M:%S' for full datetime (e.g. 2025-04-25 14:30:00), '%d/%m/%Y' for day/month/year. Defaults to '%Y-%m-%d %H:%M:%S' if omitted.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
}
)

In [33]:
messages = []

messages.append(
    {
        "role": "user",
        "content": "What is the exact time? formatted as HH:MM:SS",
    }
)

response = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools = [get_current_datetime_schema]
)
messages.append({
    "role": "assistant",
    "content": response.content
})

messages

[{'role': 'user', 'content': 'What is the exact time? formatted as HH:MM:SS'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01UFfUPa7TUt65rEmcnTBLn2', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [34]:
result = get_current_datetime(**response.content[0].input)

In [35]:
messages.append({
    "role": "user",
    "content": [
        {
        "type": "tool_result",
        "tool_use_id": response.content[0].id,
        "content": result,
        "is_error": False

    }],
})
messages

[{'role': 'user', 'content': 'What is the exact time? formatted as HH:MM:SS'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01UFfUPa7TUt65rEmcnTBLn2', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01UFfUPa7TUt65rEmcnTBLn2',
    'content': '11:21:13',
    'is_error': False}]}]

In [36]:
tool_use_id_to_add = response.content[0].id

# Filter out any existing tool_result messages for this tool_use_id
# This ensures that only one tool_result message exists for a given tool_use_id
filtered_messages = []
for msg in messages:
    is_duplicate_tool_result = False
    if msg['role'] == 'user' and isinstance(msg['content'], list):
        for content_block in msg['content']:
            if content_block['type'] == 'tool_result' and content_block.get('tool_use_id') == tool_use_id_to_add:
                is_duplicate_tool_result = True
                break
    if not is_duplicate_tool_result:
        filtered_messages.append(msg)

messages = filtered_messages

# Now, add the correct tool_result message
messages.append({
    "role": "user",
    "content": [
        {
            "type": "tool_result",
            "tool_use_id": tool_use_id_to_add,
            "content": result,
            "is_error": False
        }
    ]
})

client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = messages,
    tools = [get_current_datetime_schema]
)

Message(id='msg_01L9G8qqCPU5D926jEzGBSnV', container=None, content=[TextBlock(citations=None, text='The exact time is **11:21:13** (HH:MM:SS format).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=854, output_tokens=23, server_tool_use=None, service_tier='standard'))

In [27]:
messages = []


In [43]:
def run_tool(tool_name, tool_input):
  if tool_name == "get_current_datetime":
    return get_current_datetime(**tool_input)

  elif tool_name == "add_duration_to_datetime":
    return add_duration_to_datetime(**tool_input)

  elif tool_name == "set_reminder":
    return set_reminder(**tool_input)


def run_tools(message):
  tool_requests = [
      block for block in message.content if block.type == "tool_use"
  ]


  tool_result_blocks = []

  for tool_request in tool_requests:
      try:
        tool_output = run_tool(tool_request.name, tool_request.input)
        tool_result_block = {
            "type": "tool_result",
            "tool_use_id": tool_request.id,
            "content": json.dumps(tool_output),
            "is_error": False,
        }

      except Exception as e:
        tool_result_block = {
            "type": "tool_result",
            "tool_use_id": tool_request.id,
            "content": str(e),
            "is_error": True,
        }
      tool_result_blocks.append(tool_result_block)

  return tool_result_blocks

In [50]:
def run_conversation(messages):
  while True:
    response = chat(messages, tools=[
        get_current_datetime_schema,
        add_duration_to_datetime_schema,
        set_reminder_schema
      ])

    # Fix: Pass response.content (the list of content blocks) instead of the entire response object.
    add_assistant_message(messages, response.content)
    print(text_from_message(response))

    if response.stop_reason != "tool_use":
      break

    tool_results = run_tools(response)
    add_user_message(messages, tool_results)

  return messages

In [51]:
messages = []

add_user_message(
    messages,
    "Set a reminder for my doctor's appointment, Its 177 days after Jan 1st, 2030"
    )

run_conversation(messages)

I'll calculate when 177 days after January 1st, 2030 is, and then set a reminder for you.
Now I'll set a reminder for your doctor's appointment on that date:
----
Setting the following reminder for 2030-06-27T00:00:00:
Doctor's appointment
----
Perfect! I've set a reminder for your doctor's appointment on **Thursday, June 27, 2030** at 12:00 AM. You'll receive a notification at that time.


[{'role': 'user',
  'content': "Set a reminder for my doctor's appointment, Its 177 days after Jan 1st, 2030"},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll calculate when 177 days after January 1st, 2030 is, and then set a reminder for you.", type='text'),
   ToolUseBlock(id='toolu_01EUE77sYENSpgBVFtjovwSU', caller=DirectCaller(type='direct'), input={'datetime_str': '2030-01-01', 'duration': 177, 'unit': 'days'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01EUE77sYENSpgBVFtjovwSU',
    'content': '"Thursday, June 27, 2030 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set a reminder for your doctor's appointment on that date:", type='text'),
   ToolUseBlock(id='toolu_017QoP4VWckW7EaPfo4BGc1Z', caller=DirectCaller(type='direct'), input={'content': "Doctor's appointment", 'timestamp': '2030-06-27T

In [46]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass